In [7]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [8]:
video = cv2.VideoCapture("data/0X1A0A263B22CCD966.mp4")

if not video.isOpened():
    raise FileNotFoundError(
        f"Error: Could not open the video file'"
    )

In [9]:
while True:
    ret, frame = video.read()

    if not ret:
        print("End of video stream or cannot fetch the frame.")
        break

    if len(frame.shape) == 3:
        raw_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        raw_gray = frame

    equalized = cv2.equalizeHist(raw_gray)

    heatmap = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)

    b, g, r = cv2.split(heatmap)

    mean_b = np.mean(b)
    mean_g = np.mean(g)
    mean_r = np.mean(r)

    mean_gray = (mean_b + mean_g + mean_r) / 3.0

    scale_b = mean_gray / (mean_b if mean_b > 0 else 1)
    scale_g = mean_gray / (mean_g if mean_g > 0 else 1)
    scale_r = mean_gray / (mean_r if mean_r > 0 else 1)

    b_balanced = np.clip(b * scale_b, 0, 255).astype(np.uint8)
    g_balanced = np.clip(g * scale_g, 0, 255).astype(np.uint8)
    r_balanced = np.clip(r * scale_r, 0, 255).astype(np.uint8)

    balanced_bgr = cv2.merge([b_balanced, g_balanced, r_balanced])

    image_float = balanced_bgr.astype(np.float32)

    max_val = np.max(image_float)
    c = 255 / np.log(1 + max_val)

    log_transformed = c * np.log(1 + image_float)
    log_transformed = np.clip(log_transformed, 0, 255).astype(np.uint8)

    log_transformed_normalized = log_transformed / 255.0
    gamma = 1.5

    gamma_corrected = np.power(log_transformed_normalized, gamma)
    gamma_corrected = (gamma_corrected * 255).astype(np.uint8)

    raw_bgr = cv2.cvtColor(raw_gray, cv2.COLOR_GRAY2BGR)

    monitoring_array = np.hstack((raw_bgr, gamma_corrected))

    cv2.imshow("Ultrasound Pipeline: Raw vs Enhanced", monitoring_array)
    
    if cv2.waitKey(30) & 0xFF == ord("q"):
        break

video.release()
cv2.destroyAllWindows()

End of video stream or cannot fetch the frame.
